# ReactAgent in DemoGPT

The **ReactAgent** is a multi-turn reasoning agent that follows the **ReAct (Reasoning + Acting)** pattern. Unlike a simple tool-calling agent that makes a single tool call and returns, the ReactAgent can **chain multiple tool calls** in a loop to solve complex, multi-step tasks.

It continuously reasons about what to do next, selects and executes a tool, observes the result, and decides whether the task is complete -- repeating this cycle until the goal is achieved or a maximum iteration limit is reached.

## Setting Up Your API Key

Create a `.env` file in your project root with your OpenAI API key:

```
OPENAI_API_KEY=sk-your-key-here
```

Then load it with `python-dotenv` (already included with DemoGPT):

In [ ]:
%pip install demogpt --quiet

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")  # Loads OPENAI_API_KEY from .env file

import os
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

## How ReactAgent Works

The ReactAgent operates in a **ReAct loop** -- a cycle of reasoning and acting that repeats until the task is solved:

1. **Decision**: The agent checks whether the task is already complete based on the accumulated context. If yes, it exits the loop.
2. **Reasoning**: The agent analyzes the task and the context gathered so far, then determines which tool to use next and why.
3. **Tool Call**: The agent executes the selected tool with the appropriate arguments.
4. **Tool Result**: The output from the tool is added to the agent's context.
5. **Repeat**: Steps 1-4 repeat until the task is complete or `max_iter` is reached.

### The `max_iter` Parameter

The `max_iter` parameter (default: **10**) controls the maximum number of iterations the loop can run. This acts as a safety net to prevent infinite loops. If the agent cannot complete the task within `max_iter` iterations, it will attempt to provide the best answer it can with the context it has gathered so far.

## Basic Example

Let's start with a simple example. The agent will search for information and then perform a calculation -- two steps that require two different tools.

We give the agent:
- **TavilySearchTool**: for web searches
- **PythonTool**: for running Python code (calculations)

In [ ]:
from demogpt_agenthub.agents import ReactAgent
from demogpt_agenthub.tools import TavilySearchTool, PythonTool
from demogpt_agenthub.llms import OpenAIChatModel

llm = OpenAIChatModel(model_name="gpt-4o-mini")
agent = ReactAgent(
    tools=[TavilySearchTool(), PythonTool()],
    llm=llm,
    verbose=True,
    max_iter=10
)

response = agent.run("What is the population of France? Calculate its square root.")
print(response)

## Multi-Step Reasoning

The ReactAgent really shines when a task requires **multiple reasoning steps**. In this example, the agent needs to:
1. Search for a specific fact (the height of the Eiffel Tower)
2. Perform a calculation using that fact

The agent will automatically chain the search and calculation tools together.

In [ ]:
agent = ReactAgent(
    tools=[TavilySearchTool(), PythonTool()],
    llm=llm,
    verbose=True,
    max_iter=10
)

response = agent.run(
    "Search for the height of the Eiffel Tower in meters, "
    "then calculate how many times a 1.8m tall person would need to stack to reach that height."
)
print(response)

## Controlling Max Iterations

The `max_iter` parameter determines how many reasoning-acting cycles the agent can perform. Setting it too low may cause the agent to run out of iterations before completing a complex task. Setting it higher gives the agent more room to work but may increase cost and latency.

When `max_iter` is reached without task completion, the agent prints a **"Not Completed"** warning and tries to give the best answer it can with the context gathered so far.

Let's compare `max_iter=3` (tight limit) vs `max_iter=10` (generous limit) on a multi-step task.

In [ ]:
# With a tight iteration limit (max_iter=3)
agent_tight = ReactAgent(
    tools=[TavilySearchTool(), PythonTool()],
    llm=llm,
    verbose=True,
    max_iter=3
)

print("=== max_iter=3 ===")
response_tight = agent_tight.run(
    "What is the GDP of Germany in USD? Calculate 15% of that value and express the result in scientific notation."
)
print(f"\nFinal answer: {response_tight}")

In [ ]:
# With a generous iteration limit (max_iter=10)
agent_generous = ReactAgent(
    tools=[TavilySearchTool(), PythonTool()],
    llm=llm,
    verbose=True,
    max_iter=10
)

print("=== max_iter=10 ===")
response_generous = agent_generous.run(
    "What is the GDP of Germany in USD? Calculate 15% of that value and express the result in scientific notation."
)
print(f"\nFinal answer: {response_generous}")

## Adding Multiple Tools

The ReactAgent becomes more powerful when you give it access to a diverse set of tools. The agent will automatically decide which tool is best for each sub-task.

Here we equip it with:
- **TavilySearchTool**: Web search
- **WikipediaTool**: Wikipedia lookups
- **PythonTool**: Python code execution
- **BashTool**: Shell command execution

In [ ]:
from demogpt_agenthub.tools import TavilySearchTool, WikipediaTool, PythonTool, BashTool

agent = ReactAgent(
    tools=[TavilySearchTool(), WikipediaTool(), PythonTool(), BashTool()],
    llm=llm,
    verbose=True,
    max_iter=10
)

response = agent.run(
    "Look up Albert Einstein on Wikipedia to find his birth year, "
    "then calculate how many days have passed since his birth until January 1, 2025, "
    "and finally use bash to echo the result."
)
print(response)

## ReactAgent with RAG

You can use a **RAG (Retrieval-Augmented Generation)** instance as a tool within the ReactAgent. This lets the agent retrieve information from your own documents and combine it with other tools like PythonTool for calculations.

`BaseRAG` implements the same interface as other tools, so it integrates seamlessly into the ReactAgent's tool list.

In [ ]:
from demogpt_agenthub.rag import BaseRAG

# Create a RAG instance with some custom knowledge
rag = BaseRAG(
    llm=llm,
    vectorstore="chroma",
    persistent_path="react_rag_demo",
    index_name="demo_index",
    reset_vectorstore=True
)
rag.add_texts("The DemoGPT project has 1500 GitHub stars and was created by Melih Unsal.")

# Use RAG as a tool alongside PythonTool
agent = ReactAgent(
    tools=[rag, PythonTool()],
    llm=llm,
    verbose=True
)

response = agent.run("How many GitHub stars does DemoGPT have? Calculate the cube root of that number.")
print(response)

## Verbose Output Explained

When `verbose=True`, the ReactAgent prints color-coded output so you can follow the agent's reasoning process step by step:

| Step | Color | Description |
|------|-------|-------------|
| **Decision** | Magenta | Whether the agent considers the task complete (`True`/`False`) |
| **Reasoning** | Blue | The agent's explanation of what it plans to do next and why |
| **Tool call** | Yellow | The name of the tool being called |
| **Tool args** | Yellow | The arguments passed to the tool |
| **Tool result** | Green | The output returned by the tool |
| **Not Completed** | Red | Warning shown when `max_iter` is reached without task completion |
| **Answer** | Green | The final answer generated by the agent |

This makes it easy to debug and understand the agent's decision-making process.

## Summary

### When to Use ReactAgent

- **Multi-step tasks**: When solving a problem requires chaining multiple tools (e.g., search then calculate).
- **Complex reasoning**: When the agent needs to iteratively refine its approach based on intermediate results.
- **Dynamic workflows**: When you cannot predict ahead of time which tools will be needed or in what order.

### Best Practices

1. **Choose the right `max_iter`**: Start with the default (10) and adjust based on task complexity. Simple tasks may only need 3-5 iterations, while complex research tasks may need more.
2. **Provide diverse tools**: Give the agent access to tools that complement each other (e.g., search + compute + file I/O).
3. **Use `verbose=True` during development**: The color-coded output helps you understand and debug the agent's behavior.
4. **Combine with RAG**: Use `BaseRAG` as a tool to let the agent query your own knowledge base alongside other capabilities.
5. **Keep tasks focused**: While the ReactAgent can handle complex queries, breaking very large tasks into smaller sub-tasks often yields better results.